In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.chdir("drive/My Drive/")

In [3]:
import sys, os, json, zipfile
from pathlib import Path
from io import TextIOWrapper
from typing import List, Dict, Any, Iterator, Tuple, Optional
import requests
import re
import pandas as pd

## CVE + EPSS

In [ ]:
#!/usr/bin/env python3
"""
Adds cve2cwe.csv and removes REJECTED CVEs:
- Drops any CVE where description_en starts with "REJECTED" (case-insensitive)
  (also handles "** REJECT **" variant)
- Keeps cve2cwe.csv in sync by excluding removed CVEs
"""

# ============== CONFIG ==============
IN_ROOT      = Path(".") ### change the location to where you put the zip containing nvd files
OUT_CSV      = "cve_node.csv"
OUT_CVE2CWE_CSV = "HAS_CWE.csv"
EPSS_ENABLE  = True
EPSS_TIMEOUT = 30
EPSS_BATCH   = 100
# ====================================

# ---------- zip reader ----------
def iter_zip_json(zpath: Path) -> Iterator[Dict[str, Any]]:
    with zipfile.ZipFile(zpath, 'r') as z:
        for n in z.namelist():
            if n.lower().endswith('.json'):
                with z.open(n) as jf:
                    yield json.load(TextIOWrapper(jf, encoding="utf-8"))

# ---------- helpers ----------
def first_or_none(lst, pred=lambda x: True):
    for x in lst or []:
        if pred(x):
            return x
    return None

def extract_descriptions_en(cve_obj: dict) -> Optional[str]:
    d = first_or_none(cve_obj.get("descriptions", []), lambda x: x.get("lang") == "en")
    return (d or {}).get("value")

def extract_references(cve_obj: dict) -> str:
    seen, urls = set(), []
    for r in cve_obj.get("references", []):
        u = r.get("url")
        if u and u not in seen:
            seen.add(u)
            urls.append(u)
    return "|".join(urls)

def pick_cvss_block(metrics: dict):
    v31 = first_or_none(metrics.get("cvssMetricV31"), lambda m: True)
    if v31: return "v31", v31.get("cvssData", {}), v31.get("exploitabilityScore"), v31.get("impactScore")
    v30 = first_or_none(metrics.get("cvssMetricV30"), lambda m: True)
    if v30: return "v30", v30.get("cvssData", {}), v30.get("exploitabilityScore"), v30.get("impactScore")
    v2  = first_or_none(metrics.get("cvssMetricV2"),  lambda m: True)
    if v2:  return "v2",  v2.get("cvssData",  {}), v2.get("exploitabilityScore"), v2.get("impactScore")
    return "none", None, None, None

def normalize_cvss_fields(version_tag: str, cvssData: Optional[dict]) -> dict:
    out = {
        "cvss_version": None, "vectorString": None, "baseScore": None, "baseSeverity": None,
        "attackVector": None, "attackComplexity": None
    }
    if not cvssData:
        return out
    if version_tag in ("v31", "v30"):
        out.update({
            "cvss_version": cvssData.get("version"),
            "vectorString": cvssData.get("vectorString"),
            "baseScore": cvssData.get("baseScore"),
            "baseSeverity": cvssData.get("baseSeverity"),
            "attackVector": cvssData.get("attackVector"),
            "attackComplexity": cvssData.get("attackComplexity"),
        })
    elif version_tag == "v2":
        out.update({
            "cvss_version": cvssData.get("version"),
            "vectorString": cvssData.get("vectorString"),
            "baseScore": cvssData.get("baseScore"),
            "baseSeverity": cvssData.get("baseSeverity"),
            "attackVector": cvssData.get("accessVector"),
            "attackComplexity": cvssData.get("accessComplexity"),
        })
    return out

def extract_cpe_matches(cve_obj: dict) -> List[str]:
    """
    Walks `configurations` -> nodes -> cpeMatch and collects the 'criteria' strings.
    Returns a de-duplicated, order-preserving list of criteria (e.g. cpe:2.3:...).
    """
    cpes = []
    seen = set()
    for cfg in cve_obj.get("configurations", []) or []:
        for node in cfg.get("nodes", []) or []:
            for cm in node.get("cpeMatch", []) or []:
                crit = cm.get("criteria")
                if crit and crit not in seen:
                    seen.add(crit)
                    cpes.append(crit)
            # Some NVD JSON variants put matches under "children" or nested nodes;
            # handle nested nodes recursively if present:
            for child in node.get("children", []) or []:
                for cm in child.get("cpeMatch", []) or []:
                    crit = cm.get("criteria")
                    if crit and crit not in seen:
                        seen.add(crit)
                        cpes.append(crit)
    return cpes


def extract_cwe_ids(cve_obj: dict) -> List[str]:
    """
    Pull CWE IDs from weaknesses[].description[].value.
    Keeps values that look like 'CWE-...' (case-insensitive) and uppercases the prefix.
    """
    import re
    cwes: List[str] = []
    for w in cve_obj.get("weaknesses", []) or []:
        for d in w.get("description", []) or []:
            val = str(d.get("value", "")).strip()
            if not val:
                continue
            parts = [p.strip() for p in re.split(r"[;,]", val) if p.strip()]
            for p in parts:
                if p.upper().startswith("CWE-"):
                    cwes.append(p.upper())
    # de-dup while preserving order
    seen = set(); out = []
    for x in cwes:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

def flatten_nvd_object(obj: dict, source_hint: str) -> Tuple[List[dict], List[dict]]:
    """
    Returns:
      - rows:   core CVE rows for cve.csv (now includes a 'cpe' column)
      - cwe_rows: mappings (cve_id, cwe_id) for cve2cwe.csv
    """
    rows: List[dict] = []
    cwe_rows: List[dict] = []
    for v in obj.get("vulnerabilities", []):
        cve = v.get("cve", {})
        cve_id = cve.get("id")
        if not cve_id:
            continue

        desc_en = extract_descriptions_en(cve)
        refs    = extract_references(cve)

        # CVSS block as before
        metrics = cve.get("metrics", {}) or {}
        ver_tag, cvssData, expl_score, imp_score = pick_cvss_block(metrics)
        cvss = normalize_cvss_fields(ver_tag, cvssData)

        # NEW: extract cpe criteria (joined by | if multiple)
        cpe_list = extract_cpe_matches(cve)
        cpe_val = "|".join(cpe_list) if cpe_list else ""

        rows.append({
            "cve_id": cve_id,
            "description_en": desc_en,
            "cvss_version": cvss["cvss_version"],
            "vectorString": cvss["vectorString"],
            "baseScore": cvss["baseScore"],
            "baseSeverity": cvss["baseSeverity"],
            "attackVector": cvss["attackVector"],
            "attackComplexity": cvss["attackComplexity"],
            "exploitabilityScore": expl_score,
            "impactScore": imp_score,
            "references": refs,
            "source_hint": source_hint,
            "cpe": cpe_val,               # <-- NEW column
        })

        for cwe in extract_cwe_ids(cve):
            cwe_rows.append({"cve_id": cve_id, "cwe_id": cwe})

    return rows, cwe_rows


def fetch_epss_for_cves(cves: List[str], timeout: int = EPSS_TIMEOUT, batch: int = EPSS_BATCH) -> pd.DataFrame:
    base = "https://api.first.org/data/v1/epss"
    out = []
    sess = requests.Session()
    for i in range(0, len(cves), batch):
        chunk = cves[i:i+batch]
        try:
            r = sess.get(base, params={"cve": ",".join(chunk)}, timeout=timeout)
            r.raise_for_status()
            payload = r.json()
            for item in payload.get("data", []):
                out.append({
                    "cve_id": item.get("cve"),
                    "epss": float(item.get("epss")) if item.get("epss") else None,
                    "epss_percentile": float(item.get("percentile")) if item.get("percentile") else None,
                    "epss_date": item.get("date"),
                })
        except Exception as e:
            print(f"[WARN] EPSS batch failed ({chunk[:3]}..., {len(chunk)} CVEs): {e}", file=sys.stderr)
    return pd.DataFrame(out).drop_duplicates(subset=["cve_id"])

# ---------- main ----------
import re  # for extract_cwe_ids splitting

nvd_dir = Path(IN_ROOT) / "nvd"
if not nvd_dir.exists():
    print("NVD not found", nvd_dir, file=sys.stderr)

all_rows: List[dict] = []
all_cwe_rows: List[dict] = []

for p in nvd_dir.rglob("*"):
    if not p.is_file():
        continue
    low = p.name.lower()
    try:
        if low.endswith(".zip"):
            for obj in iter_zip_json(p):
                rows, cwe_rows = flatten_nvd_object(obj, source_hint=f"{p}")
                all_rows.extend(rows)
                all_cwe_rows.extend(cwe_rows)
        elif low.endswith(".json"):
            with open(p, "r", encoding="utf-8") as fh:
                obj = json.load(fh)
            rows, cwe_rows = flatten_nvd_object(obj, source_hint=str(p))
            all_rows.extend(rows)
            all_cwe_rows.extend(cwe_rows)
    except Exception as e:
        print(f"[WARN] Failed on {p}: {e}", file=sys.stderr)

df = pd.DataFrame(all_rows).drop_duplicates(subset=["cve_id"]).reset_index(drop=True)
if df.empty:
    print("No CVE records found.", file=sys.stderr)

# --- NEW: Drop REJECTED CVEs ---
def _is_rejected(s: Optional[str]) -> bool:
    if not isinstance(s, str):
        return False
    t = s.strip()
    up = t.upper()
    return up.startswith("REJECTED") or up.startswith("** REJECT **") or up.startswith("**REJECT**")

before = len(df)
df = df[~df["description_en"].apply(_is_rejected)].reset_index(drop=True)
removed = before - len(df)
if removed:
    print(f"Filtered out {removed} REJECTED CVEs")

# EPSS merge (on filtered set)
if EPSS_ENABLE and not df.empty:
    try:
        epss_df = fetch_epss_for_cves(df["cve_id"].dropna().unique().tolist())
        df = df.merge(epss_df, on="cve_id", how="left")
    except Exception as e:
        print(f"[WARN] EPSS lookup failed: {e}", file=sys.stderr)
        df["epss"] = None
        df["epss_percentile"] = None
        df["epss_date"] = None

# Write main CVE CSV
if OUT_CSV:
    df.to_csv(OUT_CSV, index=False, encoding="utf-8")
    print(f"Wrote {len(df)} rows → {OUT_CSV}")

# ----- cve2cwe.csv (exclude REJECTED CVEs) -----
cwe_df = pd.DataFrame(all_cwe_rows)
if not cwe_df.empty:
    cwe_df["cve_id"] = cwe_df["cve_id"].astype(str).str.strip().str.upper()
    cwe_df["cwe_id"] = cwe_df["cwe_id"].astype(str).str.strip().str.upper()
    cwe_df = cwe_df[cwe_df["cwe_id"].str.startswith("CWE-")]

    # keep only CVEs that survived the REJECT filter
    valid = set(df["cve_id"].str.upper())
    cwe_df = cwe_df[cwe_df["cve_id"].isin(valid)]

    cwe_df = cwe_df.drop_duplicates().sort_values(["cve_id", "cwe_id"]).reset_index(drop=True)
    cwe_df.to_csv(OUT_CVE2CWE_CSV, index=False, encoding="utf-8")
    print(f"Wrote {len(cwe_df)} rows → {OUT_CVE2CWE_CSV}")
else:
    print("No CVE→CWE mappings found; cve2cwe.csv not written.")


## CPE

In [11]:
#!/usr/bin/env python3
"""
Reads 'cve.csv' (must contain CVE + CPE columns), explodes pipe-separated CPEs
into one row each, derives 'type' from the CPE 2.3 part (a/h/o), and writes
'cve_cpe_type.csv' with columns: cve_id, cpe, type.

Adjustments:
- Removes any double quotes (") present in the CPE strings before processing/saving.
- No CLI arguments: reads 'cve.csv' and writes 'cve_cpe_type.csv' in the CWD.
"""

import sys
from pathlib import Path
import pandas as pd

IN_FILE = Path("cve.csv") ## the location where you put previous file
OUT_FILE = Path("cve_cpe_type.csv")

def detect_col(df: pd.DataFrame, candidates: list[str], fallback_idx: int) -> str:
    """Pick a column by name (case-insensitive); fallback to index if not found."""
    lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in lower:
            return lower[cand]
    return df.columns[fallback_idx]

def cpe_type(cpe: str) -> str:
    """Return human-friendly type from CPE 2.3 part (a/h/o)."""
    parts = (cpe or "").split(":")
    kind = parts[2].lower() if len(parts) > 2 else ""
    return {"a": "Application", "h": "Hardware", "o": "Operating System"}.get(kind, "Unknown")

def main():
    if not IN_FILE.exists():
        sys.exit(f"Input not found: {IN_FILE.as_posix()}")

    # Read
    df = pd.read_csv(IN_FILE, dtype=str).fillna("")

    # Identify column names for CVE and CPE
    cve_col = detect_col(df, ["cve_id", "cve", "id"], 0)
    cpe_col = detect_col(df, ["cpe", "criteria", "cpe_uri"], 1)

    # Work on just those two columns
    work = df[[cve_col, cpe_col]].copy()
    work.columns = ["cve_id", "cpe_raw"]

    # Strip any double quotes from CPE values before splitting
    work["cpe_raw"] = work["cpe_raw"].astype(str).str.replace('"', "", regex=False).str.strip()

    # Split 'cpe' on '|' and explode to one row per CPE
    work["cpe_list"] = work["cpe_raw"].str.split(r"\s*\|\s*", regex=True)
    exploded = (
        work.explode("cpe_list", ignore_index=True)
            .rename(columns={"cpe_list": "cpe"})
            .drop(columns=["cpe_raw"])
    )

    # Clean rows: remove any lingering quotes/spaces
    exploded["cpe"] = exploded["cpe"].astype(str).str.replace('"', "", regex=False).str.strip()

    # Drop empties / 'nan' / 'none'
    exploded = exploded[
        (exploded["cpe"] != "") &
        (~exploded["cpe"].str.lower().isin(["nan", "none"]))
    ].copy()

    # Add 'type'
    exploded["type"] = exploded["cpe"].apply(cpe_type)

    # Deduplicate and save
    out_df = exploded[["cve_id", "cpe", "type"]].drop_duplicates().reset_index(drop=True)
    out_df.to_csv(OUT_FILE, index=False, encoding="utf-8")
    print(f"Wrote {len(out_df):,} rows -> {OUT_FILE.as_posix()}")

if __name__ == "__main__":
    main()


Wrote 2,645,530 rows -> cve_cpe_type.csv


In [ ]:
cve_df = pd.read_csv("cve.csv")

def clean_desc(s: str) -> str:
    s = str(s)
    s = s.replace('"', "'")                          # 1) " -> '
    s = re.sub(r"\s*[\r\n]+\s*", " ", s)             # 2) remove newline/CR blocks
    s = re.sub(r"\s{2,}", " ", s).strip()            # 3) collapse extra spaces
    return s

cve_df["description_en"] = cve_df["description_en"].map(clean_desc)

# df is now cleaned in memory. Example peek:
print(cve_df[["cve_id", "description_en"]].head(3).to_string(index=False))

       cve_id                                                                                                                                                                                                                                                                                                                                         description_en
CVE-1999-0468                                                                                                                                                                                                              Internet Explorer 5.0 allows a remote server to read arbitrary files on the client's file system using the Microsoft Scriptlet Component.
CVE-2013-5714 Multiple cross-site scripting (XSS) vulnerabilities in ls/htmlchat.php in the VideoWhisper Live Streaming Integration plugin 4.25.3 and possibly earlier for WordPress allow remote attackers to inject arbitrary web script or HTML via the (1) name or (2) message parameter. 

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
model_id = "sarahwei/MITRE-v16-tactic-bert-case-based"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    # device_map="auto",
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


## CREATED RELATEION FOR SUGGESTED_TACTIC

In [ ]:
device = (
    "cuda" if torch.cuda.is_available()
    else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
)
model.to(device)
model.eval()

THRESHOLD = 0.5
BATCH_SIZE = 16  # adjust if you want

id2label = model.config.id2label  # index -> label string
num_labels = model.config.num_labels

all_preds = []

with torch.no_grad():
    # iterate in batches for speed/memory safety
    for start in range(0, len(cve_df), BATCH_SIZE):
        batch_texts = cve_df["description_en"].iloc[start:start+BATCH_SIZE].fillna("").tolist()

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        )
        # move tensors to device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits)                # (batch, num_labels)
        preds = (probs >= THRESHOLD).detach().cpu().numpy()  # bools

        # convert each row of bools to list of label names
        for row in preds:
            labels = [id2label[i] for i, flag in enumerate(row) if flag]
            all_preds.append(labels)

## add colum predicted labels
cve_df["predicted_labels"] = all_preds

# quick peek
print(cve_df[["cve_id", "predicted_labels"]].head().to_string(index=False))


       cve_id                    predicted_labels
CVE-1999-0468                 [TA0009:Collection]
CVE-2013-5714                                  []
CVE-2014-0752                                  []
CVE-2014-0753 [TA0003:Persistence, TA0040:Impact]
CVE-2014-0750                  [TA0002:Execution]


In [ ]:
cve_df.to_csv("cve_predictedtactic.csv", index=False, encoding="utf-8")

## UPDATE with KEV

In [ ]:
cve_df = pd.read_csv("cve_predictedtactic.csv")
# Adds a boolean "hasKEV" column to cve_predictedtactic.csv in-place.
# "hasKEV" is True if the CVE appears in CISA's Known Exploited Vulnerabilities feed.

import json
import pandas as pd
from urllib.request import urlopen

CSV_PATH = "cve_predictedtactic+kev.csv"
KEV_URL = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"

# Load CSV
df = cve_df

# Find CVE column
cve_col = None
for c in ["cve_id", "CVE", "cve", "cveID", "cveId", "CVE_ID"]:
    if c in df.columns:
        cve_col = c
        break
if cve_col is None:
    raise ValueError("Couldn't find a CVE column (tried cve_id/CVE/cve/cveID/cveId/CVE_ID)")

# Load KEV set
with urlopen(KEV_URL, timeout=30) as resp:
    kev = json.load(resp)
kev_cves = {
    str(v.get("cveID", "")).strip().upper()
    for v in kev.get("vulnerabilities", [])
    if isinstance(v, dict) and v.get("cveID")
}

# Add column: True if in KEV, else False
df["hasKEV"] = df[cve_col].astype(str).str.strip().str.upper().isin(kev_cves)

# Save in place
df.to_csv(CSV_PATH, index=False)

